In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
train_trans = pd.read_csv("D:/project/data/train_transaction.csv")
train_id = pd.read_csv("D:/project/data/train_identity.csv")
test_trans = pd.read_csv("D:/project/data/test_transaction.csv")
test_id = pd.read_csv("D:/project/data/test_identity.csv")

#merge to 1
test_df = test_trans.merge(test_id,on="TransactionID",how="left")
train_df=train_trans.merge(train_id,on="TransactionID",how="left")




# Dataset Cleaning and Preprocessing

This section describes the data preprocessing pipeline applied to the IEEE-CIS Fraud Detection dataset, including the following steps:

- Remove the label column **`isFraud`** from the feature set.
- Identify and handle missing values.
- Encode categorical features into numerical representations.
- Ensure that training and testing datasets share the same feature columns.
- Split the dataset into training and validation sets based on transaction time.


In [12]:
y = train_df["isFraud"].copy()
X = train_df.drop(columns=["isFraud"]).copy()
X_test = test_df.copy()

print("X shape     :", X.shape)
print("y shape     :", y.shape)
print("X_test shape:", X_test.shape)

X shape     : (590540, 433)
y shape     : (590540,)
X_test shape: (506691, 433)


In [13]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

X_train: (472432, 433)
X_val  : (118108, 433)
X_test : (506691, 433)


In [14]:
print("Train distribution:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

print("\nValidation distribution:")
print(y_val.value_counts())
print(y_val.value_counts(normalize=True))

Train distribution:
isFraud
0    455902
1     16530
Name: count, dtype: int64
isFraud
0    0.965011
1    0.034989
Name: proportion, dtype: float64

Validation distribution:
isFraud
0    113975
1      4133
Name: count, dtype: int64
isFraud
0    0.965007
1    0.034993
Name: proportion, dtype: float64


In [15]:
X_train, X_val = X_train.align(X_val, join="left", axis=1, fill_value=np.nan)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=np.nan)

X_val = X_val.reindex(columns=X_train.columns, fill_value=np.nan)
X_test = X_test.reindex(columns=X_train.columns, fill_value=np.nan)

print("Aligned shapes:")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

Aligned shapes:
X_train: (472432, 433)
X_val  : (118108, 433)
X_test : (506691, 433)


In [16]:
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", len(cat_cols))
print("Numerical columns  :", len(num_cols))

Categorical columns: 31
Numerical columns  : 402


In [17]:
for col in cat_cols:
    X_train[col] = X_train[col].fillna("missing")
    X_val[col]   = X_val[col].fillna("missing")
    X_test[col]  = X_test[col].fillna("missing")

print("Filled missing values for categorical columns.")

Filled missing values for categorical columns.


In [18]:
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols].astype(str))
X_val[cat_cols]   = encoder.transform(X_val[cat_cols].astype(str))
X_test[cat_cols]  = encoder.transform(X_test[cat_cols].astype(str))

print("Categorical encoding completed.")

Categorical encoding completed.


In [19]:
X_train = X_train.apply(pd.to_numeric, errors="coerce").fillna(0)
X_val   = X_val.apply(pd.to_numeric, errors="coerce").fillna(0)
X_test  = X_test.apply(pd.to_numeric, errors="coerce").fillna(0)

print("Converted all features to numeric.")
print("Any NaN in X_train:", X_train.isna().sum().sum())
print("Any NaN in X_val  :", X_val.isna().sum().sum())
print("Any NaN in X_test :", X_test.isna().sum().sum())

Converted all features to numeric.
Any NaN in X_train: 0
Any NaN in X_val  : 0
Any NaN in X_test : 0


In [20]:
train_processed = X_train.copy()
train_processed["isFraud"] = y_train.values

val_processed = X_val.copy()
val_processed["isFraud"] = y_val.values

test_processed = X_test.copy()

In [21]:
train_processed.to_csv("train_processed.csv", index=False)
val_processed.to_csv("val_processed.csv", index=False)
test_processed.to_csv("test_processed.csv", index=False)

print("Saved files:")
print("- train_processed.csv")
print("- val_processed.csv")
print("- test_processed.csv")

Saved files:
- train_processed.csv
- val_processed.csv
- test_processed.csv


In [22]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

pos_weight = neg / pos
print("neg =", neg)
print("pos =", pos)
print("pos_weight =", pos_weight)

neg = 455902
pos = 16530
pos_weight = 27.580278281911674


In [23]:
print(train_processed.head())
print("\nTrain processed shape:", train_processed.shape)
print("Validation processed shape:", val_processed.shape)
print("Test processed shape:", test_processed.shape)

        TransactionID  TransactionDT  TransactionAmt  ProductCD  card1  card2  \
40809         3027809        1008491          100.00        2.0   6177  399.0   
285886        3272886        7008212           29.99        4.0   7900  345.0   
104256        3091256        2071522          107.95        4.0  11690  111.0   
507860        3494860       13299752          241.95        4.0   2616  327.0   
196382        3183382        4412283          117.00        4.0  13780  298.0   

        card3  card4  card5  card6  ...  id_32  id_33  id_34  id_35  id_36  \
40809   150.0    0.0  150.0    1.0  ...   24.0  118.0    3.0    1.0    0.0   
285886  150.0    2.0  224.0    2.0  ...    0.0  236.0    4.0    2.0    2.0   
104256  150.0    4.0  226.0    1.0  ...    0.0  236.0    4.0    2.0    2.0   
507860  150.0    1.0  102.0    1.0  ...    0.0  236.0    4.0    2.0    2.0   
196382  150.0    4.0  226.0    2.0  ...    0.0  236.0    4.0    2.0    2.0   

        id_37  id_38  DeviceType  DeviceInfo